In [ ]:
from datetime import datetime
from getpass import getpass

rdm_url = None
idp_name_1 = None
idp_username_1 = None
idp_password_1 = None
default_result_path = None
close_on_fail = False
transition_timeout = 60 * 1000

project_name = None

halted_binderhub_url = 'https://example.com'

In [ ]:
if rdm_url is None:
    rdm_url = input(prompt=f'RDM URL: ')
if idp_name_1 is None:
    idp_name_1 = input(prompt=f'IdP Name: ')
if idp_username_1 is None:
    idp_username_1 = input(prompt=f'Username for {idp_name_1}')
if idp_password_1 is None:
    idp_password_1 = getpass(prompt=f'Password for {idp_username_1}@{idp_name_1}')
if project_name is None:
    project_name = datetime.now().strftime('TEST-BINDERHUB-%Y%m%d%H%M')

project_url = None
project_created = False

# BinderHubアドオン 停止中のBinderHubアドオン

- サブシステム名: アドオン
- ページ/アドオン: BinderHub
- 機能分類: アカウント設定
- シナリオ名:  BinderHub停止中のBinderHubアドオン
- 用意するテストデータ: URL一覧、アカウント(既存ユーザー1: GRDM, BinderHub, GRDMは全てプロフィールを埋めていること)


In [ ]:
import tempfile

work_dir = tempfile.mkdtemp()
if default_result_path is None:
    default_result_path = work_dir
work_dir


In [ ]:
import importlib
import pandas as pd

import scripts.playwright
importlib.reload(scripts.playwright)

from scripts.playwright import *
from scripts import grdm

await init_pw_context(close_on_fail=close_on_fail, last_path=default_result_path)


## ウェブブラウザの新規プライベートウィンドウでGRDMトップページを表示する

GRDMトップページが表示されること

In [ ]:
async def _step(page):
    await page.goto(rdm_url, wait_until='networkidle')
    consent_button = page.locator('//button[text() = "同意する"]')
    if await consent_button.count():
        await consent_button.click()

await run_pw(_step)


## IdPを利用し、既存ユーザー1としてログインする

GRDMダッシュボードが表示されること

In [ ]:
async def _step(page):
    await grdm.login(page, idp_name_1, idp_username_1, idp_password_1, transition_timeout=transition_timeout)
    await grdm.expect_dashboard(page, transition_timeout=transition_timeout)

await run_pw(_step)


## ダッシュボードから「新規プロジェクト作成」をクリックする

指定したプロジェクトが存在しない場合、新規プロジェクトが作成されること

In [ ]:
async def _step(page):
    global project_created
    project_created = await grdm.ensure_project_exists(page, project_name, transition_timeout=transition_timeout)
    if project_created:
        print(f'Created project: {project_name}')
    else:
        print(f'Project already exists: {project_name}')

await run_pw(_step)


## ダッシュボードのプロジェクト一覧から作成したプロジェクトをクリックする

プロジェクトダッシュボードが表示されること

In [ ]:
async def _step(page):
    global project_url
    await page.locator(f'//*[@data-test-dashboard-item-title and text() = "{project_name}"]').click()
    await expect(page.locator('//span[@id = "nodeTitleEditable"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('//a[text() = "アドオン"]')).to_be_visible(timeout=transition_timeout)
    project_url = page.url

await run_pw(_step)


## BinderHubアドオンを有効化する

アドオン利用規約の確認ダイアログが表示されること

In [ ]:
async def _step(page):
    await grdm.enable_addon(page, 'GakuNin Federated Computing Services (Jupyter)', transition_timeout=transition_timeout)

await run_pw(_step)


## 「BinderHubを追加」ボタンをクリックする

BinderHubクライアント情報の設定ダイアログが表示されること

In [ ]:
async def _step(page):
    binder_entry = page.locator(f'//a[text() = "{halted_binderhub_url}"]')
    if await binder_entry.count():
        print('BinderHub entry already exists')
        return
    add_button = page.locator('//button[@href="#binderhubInputHost"]')
    await add_button.click()
    await expect(page.locator('//input[@name = "binderhub_url"]')).to_be_visible(timeout=transition_timeout)
    await page.locator('//input[@name = "binderhub_url"]').fill(halted_binderhub_url)
    # Tab to trigger any on-change events
    await page.locator('//input[@name = "binderhub_url"]').press('Tab')
    save_button = page.locator('//button[contains(@data-bind, "hostCompleted") and contains(text(), "保存")]')
    await expect(save_button).to_be_enabled(timeout=transition_timeout)
    await save_button.click()
    await expect(page.locator(f'//a[text() = "{halted_binderhub_url}"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)


## 追加したBinderHubのチェックボックスをクリックする

Default BinderHub URL が追加したBinderHubのURLになること

In [ ]:
async def _step(page):
    default_square = page.locator(f'//a[text() = "{halted_binderhub_url}"]/../..//i[contains(@class, "fa-square")]')
    if await default_square.count():
        await default_square.click()
        await expect(page.locator(f'//a[text() = "{halted_binderhub_url}"]/../..//i[contains(@class, "fa-check-square")]')).to_be_visible(timeout=transition_timeout)
    else:
        print('BinderHub already selected')

await run_pw(_step)


## プロジェクトダッシュボードの上部メニューから「解析」をクリックする

BinderHubアドオンページが表示されること

In [ ]:
async def _step(page):
    await page.locator('//a[contains(text(), "解析")]').click()
    open_idp_list = page.locator('//*[@id = "dropdown_img"]')
    launch_button = page.locator('//*[@data-test-binderhub-launch]')
    await expect(open_idp_list.or_(launch_button)).to_be_visible(timeout=transition_timeout)

    if await open_idp_list.is_visible():
        await grdm.login(page, idp_name_1, idp_username_1, idp_password_1, transition_timeout=transition_timeout)

    await expect(launch_button).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 解析機能ページがBinderHub停止中を示唆していることを確かめる

BinderHub停止中を案内するバナーが表示されること、解析環境を作成するボタンが無効状態となっていること


In [ ]:
async def _step(page):
    launch_button = page.locator('//*[@data-test-binderhub-launch]')
    await expect(launch_button).to_be_disabled(timeout=transition_timeout)
    banner = page.locator('//p[contains(text(), "選択されたJupyterHubは現在停止中です。現在、新規の解析環境は作成できません。")]')
    await expect(banner).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## ホスト選択ダイアログを開くボタンをクリックする

ホスト選択ダイアログが表示されること

In [ ]:
async def _step(page):
    open_host_selector = page.locator('//*[@data-test-open-host-selector-dialogue]')
    await expect(open_host_selector).to_be_visible(timeout=transition_timeout)
    await open_host_selector.click()
    await expect(page.locator('//*[text() = "ホスト選択"]')).to_be_visible(timeout=transition_timeout)
    

await run_pw(_step)

## 初期設定されるホストを選択する

ホスト選択ダイアログが閉じられること

In [ ]:
async def _step(page):
    topmost = page.locator('//*[@data-test-host-selection-option]').nth(0)
    await expect(topmost).to_be_visible(timeout=transition_timeout)
    await topmost.click()
    await page.locator('//*[@data-test-host-selector-dialogue-ok]').click()
    await expect(page.locator('//*[text() = "ホスト選択"]')).not_to_be_visible(timeout=transition_timeout)
    

await run_pw(_step)

## 解析機能ページがBinderHub停止中を示唆していないことを確かめる

BinderHub停止中を案内するバナーが非表示となること、解析環境作成ボタンが有効になっていること

In [ ]:
async def _step(page):
    launch_button = page.locator('//*[@data-test-binderhub-launch]')
    await expect(launch_button).to_be_enabled(timeout=transition_timeout)
    banner = page.locator('//p[contains(text(), "選択されたJupyterHubは現在停止中です。現在、新規の解析環境は作成できません。")]')
    await expect(banner).not_to_be_visible(timeout=transition_timeout)

await run_pw(_step)

終了処理を実施する。

In [ ]:
await finish_pw_context()

In [ ]:
!rm -fr {work_dir}